# Beyond Benford's Law: a hands-on tour of fraud-detection techniques

A companion notebook to the Sheepdog Prosperity Partners article on how numbers expose financial fraud in due diligence. Every section below is self-contained and uses only the Python standard library, so there is nothing to install. Run it top to bottom, or run any single cell on its own.

Each example uses a tiny, made-up dataset chosen to trip its test, so you can see what a flag looks like. On real books these run across thousands of rows, and the real power is convergence: a transaction caught by several independent tests at once is the signal worth chasing.

Prepared by Noah Green, CPA, CFE. General educational material, not an audit method or advice.

## Reading the digits themselves

These tests do not look at what a number means, only at the digits it is made of. Real financial activity leaves a natural fingerprint in those digits that invented numbers rarely reproduce.

### Benford first-digit test

Benford's Law observes that in many natural collections of numbers, the leading digit is not uniform: a 1 appears about 30.1 percent of the time while a 9 appears under 5 percent. Genuine accounting populations such as invoice amounts or expense lines tend to follow this curve, so large gaps from the expected shares can flag invented or manipulated figures. Read the output by comparing the observed share of each leading digit against the Benford expectation, and treat a big positive spike on a single digit as a prompt to pull the underlying records.

In [1]:
import math
from collections import Counter

# Inline expense amounts. Note the heavy clustering on leading digit 5.
amounts = [512, 5230, 58, 540, 5102, 5900, 53, 5001, 587, 5400, 512, 59, 7321, 980, 134, 521]

def benford_first_digit(values):
    expected = {d: math.log10(1 + 1.0 / d) for d in range(1, 10)}
    digits = [int(str(abs(v)).lstrip("0")[0]) for v in values if abs(v) >= 1]
    n = len(digits)
    counts = Counter(digits)
    worst_digit, worst_gap = None, 0.0
    for d in range(1, 10):
        obs = counts.get(d, 0) / n
        exp = expected[d]
        gap = obs - exp
        print("digit %d: observed %5.1f%%  expected %5.1f%%  gap %+5.1f%%" % (d, obs * 100, exp * 100, gap * 100))
        if gap > worst_gap:
            worst_gap, worst_digit = gap, d
    return worst_digit, worst_gap

wd, wg = benford_first_digit(amounts)
if wg > 0.10:
    print("-> REVIEW: leading digit %d is %.1f%% over the Benford expectation, pull those records" % (wd, wg * 100))
else:
    print("-> looks clean")


digit 1: observed   6.2%  expected  30.1%  gap -23.9%
digit 2: observed   0.0%  expected  17.6%  gap -17.6%
digit 3: observed   0.0%  expected  12.5%  gap -12.5%
digit 4: observed   0.0%  expected   9.7%  gap  -9.7%
digit 5: observed  81.2%  expected   7.9%  gap +73.3%
digit 6: observed   0.0%  expected   6.7%  gap  -6.7%
digit 7: observed   6.2%  expected   5.8%  gap  +0.5%
digit 8: observed   0.0%  expected   5.1%  gap  -5.1%
digit 9: observed   6.2%  expected   4.6%  gap  +1.7%
-> REVIEW: leading digit 5 is 73.3% over the Benford expectation, pull those records


### First-two-digits and last-two-digits tests

These are refinements of the basic Benford check that look at finer slices of each number. The first-two-digits test (combinations 10 through 99) catches narrow bands of suspicious values, for example a flood of charges that all begin with the same two digits just under an approval limit. The last-two-digits test instead expects an even spread across 00 through 99, so spikes there reveal rounded, estimated, or fabricated amounts. Read both as deviation scans: an outsized count on one two-digit group is the flag.

In [2]:
from collections import Counter

# Many charges begin "49" (sitting just under a 5000 approval limit) and end in "00".
amounts = [4900, 4950, 4900, 4988, 4925, 4900, 1340, 4975, 2200, 4900, 8731, 4950, 4900, 612, 4900, 5310]

def two_digit_tests(values):
    firsts = Counter(int(str(abs(v))[:2]) for v in values if abs(v) >= 10)
    lasts = Counter(abs(v) % 100 for v in values if abs(v) >= 10)
    n = sum(firsts.values())
    top_first, top_first_n = firsts.most_common(1)[0]
    top_last, top_last_n = lasts.most_common(1)[0]
    print("most common first-two digits: %02d appears %d of %d (%.0f%%)" % (top_first, top_first_n, n, 100 * top_first_n / n))
    print("most common last-two digits:  %02d appears %d of %d (%.0f%%)" % (top_last, top_last_n, n, 100 * top_last_n / n))
    return top_first, top_first_n / n, top_last, top_last_n / n

tf, tf_share, tl, tl_share = two_digit_tests(amounts)
if tf_share > 0.20 or tl_share > 0.20:
    print("-> REVIEW: clustering on first-two %02d and last-two %02d suggests limit-skirting or rounding" % (tf, tl))
else:
    print("-> looks clean")


most common first-two digits: 49 appears 11 of 16 (69%)
most common last-two digits:  00 appears 7 of 16 (44%)
-> REVIEW: clustering on first-two 49 and last-two 00 suggests limit-skirting or rounding


### Round-number bias

People who invent numbers reach for tidy round figures far more often than real transactions produce them. Genuine invoices carry odd cents and irregular totals, while estimates and fabrications cluster on multiples of 100, 500, or 1000. This test measures the share of amounts that land on round boundaries and compares it to what a normal ledger would show. A high round-number share is the signal to question whether the entries reflect real activity.

In [3]:
# A ledger stuffed with suspiciously tidy totals.
amounts = [5000, 1000, 250.75, 2000, 500, 10000, 3000, 487.12, 1500, 2500, 7000, 642.38, 4000, 500, 6000, 1000]

def round_number_bias(values, base=500):
    n = len(values)
    round_hits = [v for v in values if float(v) % base == 0]
    share = len(round_hits) / n
    print("total amounts: %d" % n)
    print("amounts that are exact multiples of %d: %d (%.0f%%)" % (base, len(round_hits), share * 100))
    print("sample round values: %s" % sorted(set(round_hits))[:5])
    return share

share = round_number_bias(amounts)
# A clean transactional ledger rarely exceeds roughly 20 percent perfectly round amounts.
if share > 0.30:
    print("-> REVIEW: %.0f%% of entries are perfectly round, far above a normal ledger" % (share * 100))
else:
    print("-> looks clean")


total amounts: 16
amounts that are exact multiples of 500: 13 (81%)
sample round values: [500, 1000, 1500, 2000, 2500]
-> REVIEW: 81% of entries are perfectly round, far above a normal ledger


### Relative size factor

The relative size factor (RSF) test compares the largest value in a group to the second largest by dividing one by the other. In a stable, well behaved group of similar transactions that ratio stays small, so a very large RSF means one record dwarfs its peers and may be a keying error, a duplicate with an extra zero, or a fraudulent outlier. Read the output as a per-group ratio: any group whose top value is many times its runner-up earns a closer look.

In [4]:
# Vendor groups: one has a single payment that towers over the rest.
groups = {
    "Vendor A": [820, 790, 805, 12500, 815],
    "Vendor B": [4200, 4350, 4100, 4400, 4250],
    "Vendor C": [60, 75, 68, 72, 70],
}

def relative_size_factor(group_map, threshold=5.0):
    flagged = []
    for name, vals in group_map.items():
        s = sorted((abs(v) for v in vals), reverse=True)
        largest, second = s[0], s[1]
        rsf = largest / second if second else float("inf")
        print("%s: largest %.0f, second %.0f, RSF %.1f" % (name, largest, second, rsf))
        if rsf > threshold:
            flagged.append((name, rsf))
    return flagged

flags = relative_size_factor(groups)
if flags:
    for name, rsf in flags:
        print("-> REVIEW: %s has an RSF of %.1f, top payment dwarfs its peers" % (name, rsf))
else:
    print("-> looks clean")


Vendor A: largest 12500, second 820, RSF 15.2
Vendor B: largest 4400, second 4350, RSF 1.0
Vendor C: largest 75, second 72, RSF 1.0
-> REVIEW: Vendor A has an RSF of 15.2, top payment dwarfs its peers


### Digit preference (Myers-style terminal-digit heaping)

When figures are eyeballed, estimated, or rounded by hand rather than measured, the final digit stops being random and piles up on favored values such as 0 and 5. The Myers blended index is a classic demographic tool that scores this terminal-digit heaping, where 0 means the last digits are evenly spread and higher values mean stronger preference. Applied to reported quantities or amounts, a high index suggests the numbers were guessed or smoothed rather than recorded from real transactions. Read it as a heaping score: the larger it is, the less the data looks organically generated.

In [5]:
from collections import Counter

# Reported headcounts that were clearly rounded to 0 and 5.
values = [200, 145, 130, 250, 175, 90, 305, 60, 415, 120, 185, 240, 95, 350, 110, 205]

def terminal_digit_heaping(nums):
    last = [abs(v) % 10 for v in nums]
    n = len(last)
    counts = Counter(last)
    expected = n / 10.0
    # Excess concentration on the favored 0 and 5 endings.
    favored = counts.get(0, 0) + counts.get(5, 0)
    favored_share = favored / n
    for d in range(10):
        print("ends in %d: %d (expected %.1f)" % (d, counts.get(d, 0), expected))
    print("share ending in 0 or 5: %.0f%% (random would be about 20%%)" % (favored_share * 100))
    return favored_share

share = terminal_digit_heaping(values)
if share > 0.40:
    print("-> REVIEW: %.0f%% of values end in 0 or 5, strong terminal-digit heaping" % (share * 100))
else:
    print("-> looks clean")


ends in 0: 9 (expected 1.6)
ends in 1: 0 (expected 1.6)
ends in 2: 0 (expected 1.6)
ends in 3: 0 (expected 1.6)
ends in 4: 0 (expected 1.6)
ends in 5: 7 (expected 1.6)
ends in 6: 0 (expected 1.6)
ends in 7: 0 (expected 1.6)
ends in 8: 0 (expected 1.6)
ends in 9: 0 (expected 1.6)
share ending in 0 or 5: 100% (random would be about 20%)
-> REVIEW: 100% of values end in 0 or 5, strong terminal-digit heaping


## Reading the structure of the books

These tests check whether the records behave the way an honest, mechanical bookkeeping process has to behave: documents in order, no impossible combinations, no duplicates.

### Duplicate-payment and split-invoice detection

This test scans payment records for the same vendor being paid the same amount within a short window, and for several smaller payments to one vendor on the same day that together exceed a normal single invoice. It catches accidental double-pays and deliberate schemes where one large purchase is sliced into pieces to dodge a higher approval tier. Read the output as a list of suspicious groupings: each flagged set names the vendor, the amounts, and why it tripped, so a reviewer can pull the underlying invoices and confirm whether they are genuinely separate.

In [6]:
from collections import defaultdict
from datetime import date

# Each tuple: (vendor, amount, invoice_no, pay_date)
payments = [
    ("Acme Supply", 4200.00, "INV-1001", date(2025, 3, 1)),
    ("Acme Supply", 4200.00, "INV-1002", date(2025, 3, 3)),   # near-duplicate pay
    ("Globex Parts", 3000.00, "INV-2001", date(2025, 3, 5)),
    ("Globex Parts", 3000.00, "INV-2002", date(2025, 3, 5)),   # same-day split
    ("Globex Parts", 3500.00, "INV-2003", date(2025, 3, 5)),   # together > 9000
    ("Initech LLC", 1200.00, "INV-3001", date(2025, 3, 6)),
]

def scan_payments(rows, dup_window_days=3, split_threshold=9000.0):
    flags = []
    by_vendor_amt = defaultdict(list)
    by_vendor_day = defaultdict(list)
    for v, amt, inv, d in rows:
        by_vendor_amt[(v, amt)].append((inv, d))
        by_vendor_day[(v, d)].append((inv, amt))
    for (v, amt), items in by_vendor_amt.items():
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                if abs((items[i][1] - items[j][1]).days) <= dup_window_days:
                    flags.append("duplicate pay " + v + " " + format(amt, ".2f") + " " + items[i][0] + "/" + items[j][0])
    for (v, d), items in by_vendor_day.items():
        if len(items) > 1 and sum(a for _, a in items) > split_threshold:
            flags.append("possible split " + v + " on " + str(d) + " total " + format(sum(a for _, a in items), ".2f"))
    return flags

found = scan_payments(payments)
if found:
    print("Payment scan -> REVIEW: " + "; ".join(found))
else:
    print("Payment scan -> looks clean")


Payment scan -> REVIEW: duplicate pay Acme Supply 4200.00 INV-1001/INV-1002; duplicate pay Globex Parts 3000.00 INV-2001/INV-2002; possible split Globex Parts on 2025-03-05 total 9500.00


### Threshold bunching (clustering just under an approval limit)

This test counts how many transactions land in the narrow band just below a known control limit, such as the amount above which a second signature or extra reporting is required. An honest population of amounts is usually spread out, so an unusual pile-up in the last few percent below the limit suggests someone is sizing transactions to stay under the radar. Read the output as the share of activity sitting in the just-under band: a high concentration relative to the rest is the flag, and the reviewer should sample those items to see who set the amounts and why.

In [7]:
# Amounts submitted for approval; control limit is 10000 (second signoff above it)
amounts = [10000.0 * x for x in [0.12, 0.30, 0.55, 0.98, 0.99, 0.985, 0.97, 0.40, 0.99, 0.96]]

def threshold_bunching(values, limit=10000.0, band=0.05, alert_share=0.40):
    low = limit * (1.0 - band)
    just_under = [v for v in values if low <= v < limit]
    share = len(just_under) / len(values) if values else 0.0
    return share, len(just_under), len(values)

share, n_band, n_total = threshold_bunching(amounts)
print("Just-under-limit share = " + format(share, ".0%") + " (" + str(n_band) + " of " + str(n_total) + ")")
if share >= 0.40:
    print("Threshold check -> REVIEW: amounts bunched just under the 10,000 approval limit")
else:
    print("Threshold check -> looks clean")


Just-under-limit share = 60% (6 of 10)
Threshold check -> REVIEW: amounts bunched just under the 10,000 approval limit


### Sequence integrity (monotonic numbering: backdating, cloned, gap)

This test walks a stream of sequentially numbered documents (invoices, checks, journal entries) in date order and checks that the numbers always climb. A number that goes backwards relative to its date can signal backdating, a repeated number can signal a cloned or reused document, and a missing number can signal a deleted or hidden record. Read the output as a labelled list of breaks in the chain: each entry says whether it was an out-of-order, duplicate, or gap problem so the reviewer knows which document to retrieve.

In [8]:
from datetime import date

# (invoice_no, invoice_date) expected to rise together over time
docs = [
    (1001, date(2025, 1, 2)),
    (1002, date(2025, 1, 3)),
    (1003, date(2025, 1, 5)),
    (1003, date(2025, 1, 6)),   # cloned number
    (1006, date(2025, 1, 7)),   # gap: 1004, 1005 missing
    (1004, date(2025, 1, 8)),   # out of order (backdated look)
]

def sequence_integrity(rows):
    flags = []
    ordered = sorted(rows, key=lambda r: r[1])
    seen = set()
    prev = None
    for num, d in ordered:
        if num in seen:
            flags.append("duplicate number " + str(num) + " on " + str(d))
        seen.add(num)
        if prev is not None and num < prev:
            flags.append("out-of-order number " + str(num) + " after " + str(prev))
        prev = max(prev, num) if prev is not None else num
    nums = sorted(n for n, _ in rows)
    full = set(range(nums[0], nums[-1] + 1))
    missing = sorted(full - set(nums))
    if missing:
        flags.append("missing numbers " + ",".join(str(m) for m in missing))
    return flags

found = sequence_integrity(docs)
if found:
    print("Sequence check -> REVIEW: " + "; ".join(found))
else:
    print("Sequence check -> looks clean")


Sequence check -> REVIEW: duplicate number 1003 on 2025-01-06; out-of-order number 1004 after 1006; missing numbers 1005


### Impossible-cell check (transaction type vs account type; and self-approval)

This test applies two rules of business logic. First, it flags transactions that hit an account they should never touch, such as a payroll entry landing in a revenue account, which usually means a miscoding or a disguised transfer. Second, it flags any entry where the same person both created and approved it, which defeats separation-of-duties and is a classic fraud enabler. Read the output as a list of rule violations, each naming the offending entry and which rule it broke so the reviewer can demand a second approver or a corrected coding.

In [9]:
# Each dict is a journal entry
entries = [
    {"id": "JE-1", "txn_type": "payroll", "account": "expense", "created_by": "amy", "approved_by": "ben"},
    {"id": "JE-2", "txn_type": "sale",    "account": "revenue", "created_by": "cara", "approved_by": "dan"},
    {"id": "JE-3", "txn_type": "payroll", "account": "revenue", "created_by": "amy", "approved_by": "ben"},  # impossible cell
    {"id": "JE-4", "txn_type": "refund",  "account": "expense", "created_by": "eve", "approved_by": "eve"},  # self-approval
]

# Allowed account per transaction type
allowed = {"payroll": {"expense"}, "sale": {"revenue"}, "refund": {"expense", "revenue"}, "purchase": {"expense"}}

def impossible_cells(rows):
    flags = []
    for r in rows:
        ok = allowed.get(r["txn_type"], set())
        if r["account"] not in ok:
            flags.append(r["id"] + " " + r["txn_type"] + " posted to " + r["account"])
        if r["created_by"] == r["approved_by"]:
            flags.append(r["id"] + " self-approved by " + r["created_by"])
    return flags

found = impossible_cells(entries)
if found:
    print("Logic check -> REVIEW: " + "; ".join(found))
else:
    print("Logic check -> looks clean")


Logic check -> REVIEW: JE-3 payroll posted to revenue; JE-4 self-approved by eve


## Reading how activity is distributed and flows

These tests read the shape of a company's relationships and cash movement. Real demand is lumpy and money moves in plain directions; manufactured activity tends to be too even, too sudden, or circular.

### Rank-frequency (Zipf) check

In many natural ledgers the activity counts across counterparties follow Zipf's law: if you rank vendors from most to least active, the second-busiest tends to get about half the volume of the busiest, the third about a third, and so on, so a log-log plot of count against rank falls along a roughly straight line with a slope near -1. This test fits that slope and flags ledgers that are suspiciously flat (a slope near zero), because a fabricated or padded vendor book often spreads volume too evenly to look organic. Read the printed slope: close to -1 is normal, close to 0 means the distribution is unnaturally even and worth a look.

In [10]:
import math
from collections import Counter

def zipf_slope(transactions):
    counts = sorted(Counter(transactions).values(), reverse=True)
    xs = [math.log(i + 1) for i in range(len(counts))]
    ys = [math.log(c) for c in counts]
    n = len(xs)
    mx = sum(xs) / n
    my = sum(ys) / n
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = sum((x - mx) ** 2 for x in xs)
    return num / den if den else 0.0

# Inline ledger: volume spread far too evenly across vendors (tripwire).
ledger = (["V1"] * 12 + ["V2"] * 11 + ["V3"] * 11 + ["V4"] * 10 +
          ["V5"] * 10 + ["V6"] * 10 + ["V7"] * 9 + ["V8"] * 9)
slope = zipf_slope(ledger)
print("fitted log-log slope:", round(slope, 3))
if slope > -0.4:
    print("-> REVIEW: distribution is unnaturally flat (slope near 0), unlike a natural ledger")
else:
    print("-> looks clean")


fitted log-log slope: -0.135
-> REVIEW: distribution is unnaturally flat (slope near 0), unlike a natural ledger


### Novelty-decay check (new-counterparty rate over time)

As a real business matures, the share of transactions that involve a brand-new counterparty should taper off, since most volume settles onto a stable roster of known vendors; this is the intuition behind the coupon-collector problem and Heaps' law, where the count of distinct names grows fast early and then slows as the pool gets exhausted. This test walks the ledger in time order, splits it into quarters of the stream, and reports the rate of first-time-seen names in each quarter. The first quarter is naturally high because every name is new at the start, so the test compares the final quarter against the settled middle quarters; a late spike there can signal vendors created to absorb money near a deal or period close, so watch the last number against the middle two.

In [11]:
def novelty_by_quarter(stream):
    seen = set()
    n = len(stream)
    bounds = [n * (q + 1) // 4 for q in range(4)]
    rates = []
    start = 0
    for b in bounds:
        chunk = stream[start:b]
        new = sum(1 for name in chunk if name not in seen)
        for name in chunk:
            seen.add(name)
        rates.append(new / len(chunk) if chunk else 0.0)
        start = b
    return rates

# Ordered stream: after the early ramp the roster settles, then a late
# burst of brand-new vendors appears in the final quarter (tripwire).
stream = (["A", "B", "C", "A", "B", "C", "A", "B"] * 3 +
          ["A", "B", "C", "A", "B", "C", "A", "B"] * 3 +
          ["N1", "N2", "N3", "N4", "N5", "N6", "N7", "N8"])
rates = novelty_by_quarter(stream)
print("new-name rate by quarter:", [round(r, 2) for r in rates])
# Compare the final quarter against the settled middle quarters (Q2, Q3),
# not Q1, since Q1 is inflated by the unavoidable startup ramp.
settled = (rates[1] + rates[2]) / 2
if rates[-1] > 0.25 and rates[-1] > settled + 0.25:
    print("-> REVIEW: late burst of brand-new counterparties after novelty had settled")
else:
    print("-> looks clean")


new-name rate by quarter: [1.0, 0.0, 0.0, 0.57]
-> REVIEW: late burst of brand-new counterparties after novelty had settled


### Round-tripping / circular-flow check

Round-tripping is when money leaves the business to a counterparty and comes back from that same counterparty, a pattern that can inflate apparent revenue or disguise the true direction of cash. This test builds the set of names the business paid and the set of names it received money from, then intersects them: any name appearing in both directions is a circular-flow candidate. Read the printed overlap; a normal vendor (pure outflow) or customer (pure inflow) will not appear, so any listed name deserves a closer look at why cash moves both ways.

In [12]:
def circular_counterparties(payments_out, payments_in):
    paid_to = {p["to"] for p in payments_out}
    paid_by = {p["from"] for p in payments_in}
    return paid_to & paid_by

# Outflows and inflows; "Helio Trading" sits on both sides (tripwire).
payments_out = [{"to": "Acme Supply"}, {"to": "Helio Trading"}, {"to": "Birch Freight"}]
payments_in = [{"from": "North Retail"}, {"from": "Helio Trading"}, {"from": "East Mart"}]
overlap = circular_counterparties(payments_out, payments_in)
print("counterparties paid AND received from:", sorted(overlap))
if overlap:
    print("-> REVIEW: circular cash flow with", ", ".join(sorted(overlap)))
else:
    print("-> looks clean")


counterparties paid AND received from: ['Helio Trading']
-> REVIEW: circular cash flow with Helio Trading


### Ghost-vendor / ghost-employee cross-match

A ghost vendor (or ghost employee) is a payee on the books that shares an identifying detail with an insider, for example the same bank account, tax id, or mailing address as an employee, which is a classic route for diverting funds. This test cross-matches the vendor list against the employee list on shared bank account numbers and reports any collision. Read the printed matches; each pair means a vendor and an employee share a deposit account, which should never happen and warrants immediate review.

In [13]:
def shared_bank_accounts(vendors, employees):
    emp_by_acct = {}
    for e in employees:
        emp_by_acct.setdefault(e["bank_account"], []).append(e["name"])
    hits = []
    for v in vendors:
        for emp_name in emp_by_acct.get(v["bank_account"], []):
            hits.append((v["name"], emp_name, v["bank_account"]))
    return hits

# Vendor "Crestline LLC" shares a bank account with employee "Dana Pyle" (tripwire).
vendors = [{"name": "Crestline LLC", "bank_account": "00471920"},
           {"name": "Orchard Parts", "bank_account": "88820017"}]
employees = [{"name": "Dana Pyle", "bank_account": "00471920"},
             {"name": "Sam Ortiz", "bank_account": "55510023"}]
hits = shared_bank_accounts(vendors, employees)
for vend, emp, acct in hits:
    print("shared account", acct, "->", "vendor", vend, "and employee", emp)
if hits:
    print("-> REVIEW: vendor and employee share a bank account")
else:
    print("-> looks clean")


shared account 00471920 -> vendor Crestline LLC and employee Dana Pyle
-> REVIEW: vendor and employee share a bank account


## Reading timing and the whole-company picture

These tests read the calendar, the clock, and the financial statements as a group, looking for patterns that honest activity rarely produces.

### Period-End Cutoff and Seasonality

Honest revenue tends to spread across a period, but manipulated books often pile sales into the final few days so a quarter or year hits its target. This test counts how much of total revenue is booked in the last stretch of the period and compares it against an even baseline. A heavy last-days concentration is a classic cutoff red flag that deserves a closer look at shipping dates and contract terms.

In [14]:
import datetime

# Inline daily revenue for a single quarter (Jan 1 to Mar 31).
# A big block is dumped into the last 3 days to hit the number.
entries = []
start = datetime.date(2025, 1, 1)
for i in range(90):
    d = start + datetime.timedelta(days=i)
    amt = 1000.0
    if i >= 87:           # last 3 days of the quarter
        amt = 40000.0
    entries.append((d, amt))

total = sum(a for _, a in entries)
last_days = 3
cutoff = entries[-1][0] - datetime.timedelta(days=last_days - 1)
tail = sum(a for d, a in entries if d >= cutoff)
tail_share = tail / total
even_share = last_days / len(entries)   # what an even spread would give

print("Total revenue: {:.0f}".format(total))
print("Last {} days share: {:.1%} (even baseline {:.1%})".format(last_days, tail_share, even_share))
if tail_share > 3 * even_share:
    print("-> REVIEW: revenue heavily concentrated at period end, check cutoff and shipping dates")
else:
    print("-> looks clean")


Total revenue: 207000
Last 3 days share: 58.0% (even baseline 3.3%)
-> REVIEW: revenue heavily concentrated at period end, check cutoff and shipping dates


### Posting-Time Anomalies

Routine business entries cluster during normal working hours on weekdays. Entries posted on weekends or late at night can be innocent, but a cluster of them, especially large ones, can signal rushed adjustments or someone working when oversight is thin. This test flags any entry posted outside business hours or on a weekend and summarizes the off-hours dollar exposure.

In [15]:
import datetime

# (timestamp, amount) journal entries. Two land on a Saturday and late at night.
entries = [
    (datetime.datetime(2025, 3, 12, 10, 30), 1200.0),
    (datetime.datetime(2025, 3, 13, 14, 5), 800.0),
    (datetime.datetime(2025, 3, 15, 2, 15), 95000.0),   # Saturday, 2am
    (datetime.datetime(2025, 3, 14, 23, 50), 60000.0),  # Friday, near midnight
    (datetime.datetime(2025, 3, 17, 9, 45), 500.0),
]

def off_hours(ts):
    weekend = ts.weekday() >= 5          # 5=Sat, 6=Sun
    after_hours = ts.hour < 7 or ts.hour >= 19
    return weekend or after_hours

flagged = [(ts, amt) for ts, amt in entries if off_hours(ts)]
exposure = sum(amt for _, amt in flagged)
print("Entries: {}, off-hours entries: {}".format(len(entries), len(flagged)))
for ts, amt in flagged:
    print("  {} ({}) amount {:.0f}".format(ts, ts.strftime("%A"), amt))
if flagged:
    print("-> REVIEW: {:.0f} posted outside business hours or on a weekend".format(exposure))
else:
    print("-> looks clean")


Entries: 5, off-hours entries: 2
  2025-03-15 02:15:00 (Saturday) amount 95000
  2025-03-14 23:50:00 (Friday) amount 60000
-> REVIEW: 155000 posted outside business hours or on a weekend


### Earnings-Manipulation and Distress Models

Two well-known public models help size up a whole company. The Altman Z-score blends five balance-sheet and income ratios into a single number, where a result below 1.81 signals financial distress and above 2.99 is considered safe. The Beneish M-score is a separate published model that uses eight ratios (such as days sales in receivables and total accruals) to estimate the likelihood that earnings have been manipulated; here we name it and compute the Altman Z below. A distress-zone Z does not prove fraud, but it raises the stakes on every other finding.

In [16]:
# Altman Z-score using the published coefficients.
# Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
# Realistic figures for a struggling firm (all in thousands).
working_capital = -200.0
total_assets = 1000.0
retained_earnings = -150.0
ebit = 40.0
market_value_equity = 120.0
total_liabilities = 900.0
sales = 600.0

X1 = working_capital / total_assets
X2 = retained_earnings / total_assets
X3 = ebit / total_assets
X4 = market_value_equity / total_liabilities
X5 = sales / total_assets

Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
print("Altman Z-score: {:.2f}".format(Z))
if Z < 1.81:
    print("-> REVIEW: distress zone (Z < 1.81), high financial-distress risk")
elif Z > 2.99:
    print("-> looks clean (safe zone, Z > 2.99)")
else:
    print("-> REVIEW: grey zone (1.81 to 2.99), inconclusive")


Altman Z-score: 0.36
-> REVIEW: distress zone (Z < 1.81), high financial-distress risk


### Multivariate Outlier Detection

A single entry can look normal on size, on timing, and on frequency when each is viewed alone, yet be a clear outlier when all three are considered together. This test standardizes each feature into a z-score (how many standard deviations from the average it sits) and sums the absolute z-scores into one combined score. The row with the largest combined deviation is the one most worth a manual look.

In [17]:
import statistics

# Each row: (id, amount, postings_that_day, hour_of_day). Row 3 is jointly extreme.
rows = [
    ("E1", 1200.0, 12, 10),
    ("E2", 1500.0, 14, 11),
    ("E3", 98000.0, 1, 2),     # huge, lonely, middle of the night
    ("E4", 1100.0, 13, 14),
    ("E5", 1300.0, 15, 9),
]

def zscores(vals):
    mu = statistics.mean(vals)
    sd = statistics.pstdev(vals)
    if sd == 0:
        return [0.0 for _ in vals]
    return [(v - mu) / sd for v in vals]

amts = zscores([r[1] for r in rows])
freq = zscores([r[2] for r in rows])
hour = zscores([r[3] for r in rows])

combined = []
for i, r in enumerate(rows):
    score = abs(amts[i]) + abs(freq[i]) + abs(hour[i])
    combined.append((r[0], score))

worst = max(combined, key=lambda x: x[1])
for name, score in combined:
    print("{}: combined deviation {:.2f}".format(name, score))
if worst[1] > 3.0:
    print("-> REVIEW: {} is a joint outlier on size, frequency, and timing".format(worst[0]))
else:
    print("-> looks clean")


E1: combined deviation 0.90
E2: combined deviation 1.54
E3: combined deviation 5.77
E4: combined deviation 2.11
E5: combined deviation 1.33
-> REVIEW: E3 is a joint outlier on size, frequency, and timing


### Memo and Description Text Analytics

The free-text memo on an entry is an underused signal. This test does three things: it scans memos for risk keywords like reclass, plug, or adjust; it counts identical templated memos reused across many entries; and it flags any large-dollar entry that was posted with a blank memo. Each pattern alone can be benign, but together they point to where documentation is thin and amounts are large.

In [18]:
from collections import Counter

# (memo_text, amount)
entries = [
    ("Reclass to hit target", 50000.0),
    ("Standard monthly accrual", 1200.0),
    ("Standard monthly accrual", 1200.0),
    ("Standard monthly accrual", 1200.0),
    ("", 75000.0),                       # large entry, blank memo
    ("Customer invoice 4471", 900.0),
]

risk_words = ["reclass", "plug", "adjust", "manual", "reverse", "to hit"]
large_threshold = 20000.0
template_threshold = 3

risk_hits = []
for memo, amt in entries:
    low = memo.lower()
    if any(w in low for w in risk_words):
        risk_hits.append((memo, amt))

memo_counts = Counter(m for m, _ in entries if m.strip())
templated = [(m, c) for m, c in memo_counts.items() if c >= template_threshold]
blank_large = [(m, amt) for m, amt in entries if not m.strip() and amt >= large_threshold]

print("Risk-keyword hits: {}".format(risk_hits))
print("Templated memos (>= {} uses): {}".format(template_threshold, templated))
print("Blank memo on large entry: {}".format(blank_large))
if risk_hits or templated or blank_large:
    print("-> REVIEW: memo patterns suggest thin documentation on material entries")
else:
    print("-> looks clean")


Risk-keyword hits: [('Reclass to hit target', 50000.0)]
Templated memos (>= 3 uses): [('Standard monthly accrual', 3)]
Blank memo on large entry: [('', 75000.0)]
-> REVIEW: memo patterns suggest thin documentation on material entries


## The real edge is convergence

No single test above is proof. Each one throws some false alarms on its own. The power comes from running them together and watching where they overlap. A transaction flagged by one test is a question. The same transaction flagged by several independent tests at once is a finding, because independent structural failures rarely line up by accident.

Prepared by Noah Green, CPA, CFE. Sheepdog Prosperity Partners. This notebook is general educational material about technique, not an engagement, an audit opinion, or a promise of any specific outcome.